In [ ]:
import threading
import numpy as np
import time


In [ ]:
def multiply_section(vector, results, start_idx, end_idx, scalar):
    """Multiply a section of the vector by scalar"""
    # NumPy operation - GIL released during computation
    results[start_idx:end_idx] = vector[start_idx:end_idx] * scalar
    print(f"Multiplication thread processed indices {start_idx}-{end_idx}")


def sum_section(vector, results, start_idx, end_idx, result_lock, thread_id):
    """Sum a section of the vector and add to total"""
    # Calculate partial sum for this section
    partial_sum = np.sum(vector[start_idx:end_idx])

    # Need lock when updating shared result
    with result_lock:
        results[0] += partial_sum
    print(
        f"Sum thread {thread_id} processed indices {start_idx}-{end_idx}, partial sum: {partial_sum}"
    )


In [ ]:
if __name__ == "__main__":
    # Create a sample vector
    vector_size = 10
    vector = np.arange(1, vector_size + 1, dtype=float)
    print(f"Original vector: {vector}")

In [ ]:
if __name__ == "__main__":
    # ---------- MULTIPLICATION EXAMPLE ----------
    print("\n--- MULTIPLICATION WITH 2 THREADS ---")
    multiply_results = np.zeros_like(vector)

    # Split vector into 2 sections
    mid_point = vector_size // 2

    # Create 2 multiplication threads
    thread_mul1 = threading.Thread(
        target=multiply_section, args=(vector, multiply_results, 0, mid_point, 2)
    )
    thread_mul2 = threading.Thread(
        target=multiply_section, args=(vector, multiply_results, mid_point, vector_size, 2)
    )

    # Start threads
    start_time = time.time()
    thread_mul1.start()
    thread_mul2.start()

    # Wait for completion
    thread_mul1.join()
    thread_mul2.join()

    print(f"Multiplied vector: {multiply_results}")
    print(f"Time taken: {time.time() - start_time:.4f} seconds")

In [ ]:
if __name__ == "__main__":
    # ---------- SUM EXAMPLE ----------
    print("\n--- SUM WITH 2 THREADS ---")

    # Using list for mutable shared result (or array with single element)
    sum_result = [0.0]  # Shared mutable result
    sum_lock = threading.Lock()

    # Split vector into 2 sections
    mid_point = vector_size // 2

    # Create 2 sum threads
    thread_sum1 = threading.Thread(
        target=sum_section, args=(vector, sum_result, 0, mid_point, sum_lock, 1)
    )
    thread_sum2 = threading.Thread(
        target=sum_section, args=(vector, sum_result, mid_point, vector_size, sum_lock, 2)
    )

    # Start threads
    start_time = time.time()
    thread_sum1.start()
    thread_sum2.start()

    # Wait for completion
    thread_sum1.join()
    thread_sum2.join()

    print(f"Total sum: {sum_result[0]}")
    print(f"Verification (np.sum): {np.sum(vector)}")
    print(f"Time taken: {time.time() - start_time:.4f} seconds")
